# Sales Data Mart with the `cubes` OLAP framework

Steps:
1. Install and import `cubes`
2. Build the SQLite data mart (`sales_datamart.sqlite`)
3. Describe the cube in a model file (`sales_datamart.json`)
4. Open it with `Workspace` and a `browser`
5. Explore with `Cell`, `PointCut`, `SetCut`, `RangeCut`, drilldown, facts and members

## Step 1: Install and import

`cubes` 1.1 is an older library. On Python 3.10+ it needs SQLAlchemy below 1.4 and a small shim for `collections.Mapping`.

In [2]:
# Run once:
# !pip install cubes "sqlalchemy<1.4" pandas matplotlib

# Compatibility shim for Python 3.10+ (must run BEFORE importing cubes)
import collections, collections.abc
for name in ("Mapping", "MutableMapping", "Sequence", "Iterable", "Callable", "Set"):
    if not hasattr(collections, name):
        setattr(collections, name, getattr(collections.abc, name))

import os, json, random, sqlite3
from datetime import date, timedelta
import pandas as pd
from cubes import Workspace, Cell, PointCut, SetCut, RangeCut
print('cubes imported')

cubes imported


## Step 2: Build the data mart database

A star schema in SQLite: one fact table and three dimension tables. `profit` is stored in the fact table so the cube can sum it.

In [3]:
random.seed(42)
if os.path.exists('../data/sales_datamart.sqlite'):
    os.remove('../data/sales_datamart.sqlite')

conn = sqlite3.connect('../data/sales_datamart.sqlite')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE dim_product  (product_id INTEGER PRIMARY KEY, product_name TEXT, category TEXT, unit_cost REAL);
CREATE TABLE dim_customer (customer_id INTEGER PRIMARY KEY, customer_name TEXT, country TEXT);
CREATE TABLE dim_date     (date_id TEXT PRIMARY KEY, year INTEGER, quarter TEXT, month TEXT, month_number INTEGER);
CREATE TABLE fact_sales   (order_id INTEGER, date_id TEXT, product_id INTEGER, customer_id INTEGER,
                           quantity INTEGER, sales_amount REAL, profit REAL);
''')

products = [(1,'Laptop','Electronics',700.0), (2,'Headphones','Electronics',40.0), (3,'Phone','Electronics',400.0),
            (4,'Desk Chair','Furniture',90.0), (5,'Desk','Furniture',150.0),
            (6,'Notebook','Stationery',2.0), (7,'Pen Set','Stationery',5.0)]
customers = [(1,'Alice Smith','USA'), (2,'Bob Jones','UK'), (3,'Carla Diaz','Spain'),
             (4,'Dan Miller','USA'), (5,'Eva Rossi','Italy'), (6,'Frank Lee','UK')]
cur.executemany('INSERT INTO dim_product  VALUES (?,?,?,?)', products)
cur.executemany('INSERT INTO dim_customer VALUES (?,?,?)', customers)

d, dates = date(2025, 1, 1), []
while d <= date(2026, 12, 31):
    dates.append((d.isoformat(), d.year, f'{d.year}-Q{(d.month-1)//3+1}', d.strftime('%Y-%m'), d.month))
    d += timedelta(days=1)
cur.executemany('INSERT INTO dim_date VALUES (?,?,?,?,?)', dates)

markup = {'Electronics': 1.35, 'Furniture': 1.5, 'Stationery': 2.5}
facts = []
for order_id in range(1001, 1401):
    p, c = random.choice(products), random.choice(customers)
    qty = random.randint(1, 4)
    day = date(2025, 1, 1) + timedelta(days=random.randint(0, 600))
    amount = round(p[3] * markup[p[2]] * qty, 2)
    facts.append((order_id, day.isoformat(), p[0], c[0], qty, amount, round(amount - qty * p[3], 2)))
cur.executemany('INSERT INTO fact_sales VALUES (?,?,?,?,?,?,?)', facts)
conn.commit()
conn.close()
print('sales_datamart.sqlite created with', len(facts), 'orders')

sales_datamart.sqlite created with 400 orders


## Step 3: Describe the cube in `sales_datamart.json`

The model tells `cubes` three things:
- **dimensions** and their **levels** (the hierarchies you can drill through): date is year, quarter, month; product is category, product; customer is country, customer
- **aggregates** (what to compute): units, revenue, profit, order count
- **joins** and **mappings**: how the fact table connects to the dimension tables and which column each attribute comes from

In [4]:
print("sales_datamart.json is written")

sales_datamart.json is written


## Step 4: Open the cube

In [5]:
from cubes import Workspace

workspace=Workspace(load_base_model=False)
workspace.register_default_store("sql", url="sqlite:///../data/sales_datamart.sqlite")
workspace.import_model("../models/sales_datamart.json")

cube=workspace.cube("sales_datamart")
browser=workspace.browser("sales_datamart")

print('cube name: ',cube.name)
print("cube's dimensions: ", [d.name for d in cube.dimensions])
print("cube's measures: ", [m.name for m in cube.measures])
print("cube's aggregates: ", [a.name for a in cube.aggregates])
# Cube: sales_mart
# Dimensions: ['date', 'product', 'customer']
# Aggregates: ['quantity_sum', 'sales_amount_sum', 'profit_sum', 'order_count']

cube name:  sales_datamart
cube's dimensions:  ['dim_product', 'dim_customer', 'dim_date']
cube's measures:  ['profit', 'sales_amount', 'quantity']
cube's aggregates:  ['profit_sum', 'revenue_sum', 'quantity_sum', 'order_count']


## Step 5: Explore the cube

### 5a. Total 

In [7]:
result=browser.aggregate()
print(result.summary)

{'profit_sum': 76270.0, 'revenue_sum': 275964.0, 'quantity_sum': 1016, 'order_count': 400}


### 5b. Drilldown: break the total down by one dimension

In [11]:
def show(result):
    return pd.DataFrame(list(result)).round(0)

print("By year:")
display(show(browser.aggregate(drilldown=["dim_date.year"])))

print("By product category:")
display(show(browser.aggregate(drilldown=["dim_product.category"])))

print("By customer's country:")
display(show(browser.aggregate(drilldown=["dim_customer.country"])))


By year:


,dim_date.year,profit_sum,revenue_sum,quantity_sum,order_count
0,2025,47844.0,174064.0,625,250
1,2026,28426.0,101900.0,391,150


By product category:


,dim_product.category,profit_sum,revenue_sum,quantity_sum,order_count
0,Electronics,57169.0,220509.0,443,171
1,Furniture,17715.0,53145.0,303,116
2,Stationery,1386.0,2310.0,270,113


By customer's country:


,dim_customer.country,profit_sum,revenue_sum,quantity_sum,order_count
0,Italy,11158.0,40506.0,159,68
1,Spain,10508.0,37764.0,152,56
2,UK,26325.0,94037.0,379,142
3,USA,28279.0,103657.0,326,134


### 5c. `PointCut`: slice on ONE value

A `Cell` is a filtered region of the cube. A `PointCut` fixes a dimension to one path. The path follows the hierarchy: `[2026]` is a year, `[2026, "2026-Q1"]` is a quarter inside that year.

In [38]:
cell = Cell(cube, [PointCut("dim_date", [2026])])
print('2026 total:', )

print(browser.aggregate(cell).summary)

print('2026 by product category')

display(show(browser.aggregate(cell,drilldown=["dim_product.category"])))

2026 total:
{'profit_sum': 28425.5, 'revenue_sum': 101900.5, 'quantity_sum': 391, 'order_count': 150}
2026 by product category


,dim_product.category,profit_sum,revenue_sum,quantity_sum,order_count
0,Electronics,19943.0,76923.0,181,65
1,Furniture,8130.0,24390.0,136,52
2,Stationery,352.0,588.0,74,33


**Drill down by changing the path and the drilldown level:** year, then quarter, then month.

In [39]:

print('2026 by quarter')

display(pd.DataFrame(list(browser.aggregate(cell,drilldown=["dim_date.quarter"]))))


print('2026-Q1 by month')

cuts=[
    PointCut("dim_date",[2026,"2026-Q1"])
]
cell=Cell(cube,cuts)
display(show(browser.aggregate(cell,drilldown=["dim_date.month"])))


2026 by quarter


,dim_date.year,dim_date.quarter,profit_sum,revenue_sum,quantity_sum,order_count
0,2026,2026-Q1,11844.0,43056.0,139,55
1,2026,2026-Q2,9102.5,31885.5,145,54
2,2026,2026-Q3,7479.0,26959.0,107,41


2026-Q1 by month


,dim_date.year,dim_date.quarter,dim_date.month,profit_sum,revenue_sum,quantity_sum,order_count
0,2026,2026-Q1,2026-01,2914.0,10764.0,45,18
1,2026,2026-Q1,2026-02,4600.0,16990.0,36,14
2,2026,2026-Q1,2026-03,4330.0,15302.0,58,23


### 5d. `SetCut`: pick SEVERAL values (a dice)

Electronics and Furniture only, broken down by country. (`hierarchy="default"` is passed explicitly because `SetCut` in this version needs it.)

In [47]:

print('Electronics + Furniture total:', )

setcut=[
    SetCut("dim_product",[["Electronics"],["Furniture"]],hierarchy="default")
]
cell=Cell(cube,setcut)
display(show(browser.aggregate(cell,drilldown=["dim_customer.country"])))

Electronics + Furniture total:


,dim_customer.country,profit_sum,revenue_sum,quantity_sum,order_count
0,Italy,10889.0,40059.0,107,47
1,Spain,10336.0,37476.0,111,40
2,UK,25782.0,93132.0,276,98
3,USA,27877.0,102987.0,252,102


### 5e. `RangeCut`: a range between two points

From 2026-Q1 through 2026-Q3.

In [74]:

print('2026 Q1-Q3 total:', )

range_cut=[
    RangeCut("dim_date",[2026,"2026-Q1"],[2026,"2026-Q3"],hierarchy="default_year")
]

cell=Cell(cube,range_cut)

print(browser.aggregate(cell).summary)

display(show(browser.aggregate(cell,drilldown=[("dim_date",None,"month")])))

2026 Q1-Q3 total:
{'profit_sum': 28425.5, 'revenue_sum': 101900.5, 'quantity_sum': 391, 'order_count': 150}


,dim_date.year,dim_date.quarter,dim_date.month,profit_sum,revenue_sum,quantity_sum,order_count
0,2026,2026-Q1,2026-01,2914.0,10764.0,45,18
1,2026,2026-Q1,2026-02,4600.0,16990.0,36,14
2,2026,2026-Q1,2026-03,4330.0,15302.0,58,23
3,2026,2026-Q2,2026-04,3177.0,11107.0,51,20
4,2026,2026-Q2,2026-05,4131.0,14633.0,57,20
5,2026,2026-Q2,2026-06,1794.0,6146.0,37,14
6,2026,2026-Q3,2026-07,4024.0,14406.0,56,22
7,2026,2026-Q3,2026-08,3454.0,12554.0,51,19


### 5f. Combine cuts: a cell with several dimensions

Electronics sold to UK customers in 2026, by month.

In [87]:

print('Electronics, UK, 2026:', )

cuts=[
    PointCut("dim_product",["Electronics"]),
    PointCut("dim_customer",["UK"]),
    PointCut("dim_date",[2026])
]

cell=Cell(cube,cuts)

print(browser.aggregate(cell).summary)

display(show(browser.aggregate(cell,drilldown=[("dim_date",None,"month")])))


Electronics, UK, 2026:
{'profit_sum': 4907.0, 'revenue_sum': 18927.0, 'quantity_sum': 58, 'order_count': 19}


,dim_date.year,dim_date.quarter,dim_date.month,profit_sum,revenue_sum,quantity_sum,order_count
0,2026,2026-Q1,2026-01,1190.0,4590.0,13,4
1,2026,2026-Q1,2026-02,763.0,2943.0,5,2
2,2026,2026-Q2,2026-04,196.0,756.0,5,2
3,2026,2026-Q2,2026-05,56.0,216.0,4,1
4,2026,2026-Q2,2026-06,518.0,1998.0,10,3
5,2026,2026-Q3,2026-07,924.0,3564.0,12,4
6,2026,2026-Q3,2026-08,1260.0,4860.0,9,3


### 5g. Cross-tab with two drilldowns

In [69]:
display(show(browser.aggregate(drilldown=["dim_product.category","dim_date.year"])))

df=pd.DataFrame(browser.aggregate(drilldown=["dim_product.category","dim_date.year"]))

pd.pivot_table(df,values="revenue_sum",index="dim_product.category",columns="dim_date.year",aggfunc="sum",margins=True,margins_name="ALL").round(0)

,dim_product.category,dim_date.year,profit_sum,revenue_sum,quantity_sum,order_count
0,Electronics,2025,37226.0,143586.0,262,106
1,Electronics,2026,19943.0,76923.0,181,65
2,Furniture,2025,9585.0,28755.0,167,64
3,Furniture,2026,8130.0,24390.0,136,52
4,Stationery,2025,1034.0,1722.0,196,80
5,Stationery,2026,352.0,588.0,74,33


dim_date.year,2025,2026,ALL
dim_product.category,,,
Electronics,143586.0,76923.0,220509.0
Furniture,28755.0,24390.0,53145.0
Stationery,1722.0,588.0,2310.0
ALL,174064.0,101900.0,275964.0


### 5h. `members`: list the values of a dimension level

In [89]:
print("Product Categories and names: ")

display(show(browser.members(Cell(cube),"dim_product",depth=2)))

print("Quarters:")

display(show(browser.members(Cell(cube),"dim_date",depth=2)))

Product Categories and names: 


,dim_product.category,dim_product.product_id,dim_product.product_name
0,Electronics,1,Laptop
1,Electronics,2,Headphones
2,Electronics,3,Phone
3,Furniture,4,Desk Chair
4,Furniture,5,Desk
5,Stationery,6,Notebook
6,Stationery,7,Pen Set


Quarters:


,dim_date.year,dim_date.quarter
0,2025,2025-Q1
1,2025,2025-Q2
2,2025,2025-Q3
3,2025,2025-Q4
4,2026,2026-Q1
5,2026,2026-Q2
6,2026,2026-Q3


### 5i. `facts`: see the underlying rows behind a cell

In [85]:
cuts=[
    PointCut("dim_date",[2026,"2026-Q1"]),
    PointCut("dim_product",["Electronics"])
]


cell=Cell(cube,cuts)

display(show(browser.facts(cell,page=0,page_size=10)))

,__fact_key__,dim_product.product_id,dim_product.product_name,dim_product.category,dim_customer.customer_id,dim_customer.customer_name,dim_customer.country,dim_date.year,dim_date.quarter,dim_date.month,profit,sales_amount,quantity
0,1013,1,Laptop,Electronics,5,Eva Rossi,Italy,2026,2026-Q1,2026-01,735.0,2835.0,3
1,1023,2,Headphones,Electronics,1,Alice Smith,USA,2026,2026-Q1,2026-02,42.0,162.0,3
2,1087,3,Phone,Electronics,1,Alice Smith,USA,2026,2026-Q1,2026-01,280.0,1080.0,2
3,1105,2,Headphones,Electronics,2,Bob Jones,UK,2026,2026-Q1,2026-02,28.0,108.0,2
4,1106,1,Laptop,Electronics,2,Bob Jones,UK,2026,2026-Q1,2026-02,735.0,2835.0,3
5,1108,2,Headphones,Electronics,6,Frank Lee,UK,2026,2026-Q1,2026-01,14.0,54.0,1
6,1126,2,Headphones,Electronics,2,Bob Jones,UK,2026,2026-Q1,2026-01,56.0,216.0,4
7,1143,2,Headphones,Electronics,4,Dan Miller,USA,2026,2026-Q1,2026-03,42.0,162.0,3
8,1156,1,Laptop,Electronics,3,Carla Diaz,Spain,2026,2026-Q1,2026-02,735.0,2835.0,3
9,1159,1,Laptop,Electronics,4,Dan Miller,USA,2026,2026-Q1,2026-02,735.0,2835.0,3


## Summary

| Concept | `cubes` object |
|---|---|
| Connect | `Workspace`, `register_store`, `import_model` |
| Open | `workspace.cube(...)`, `workspace.browser(cube)` |
| Filtered region | `Cell(cube, [cuts])` |
| Slice (one value) | `PointCut` |
| Dice (several values) | `SetCut` |
| Range | `RangeCut` |
| Roll-up / drill-down | `drilldown=[...]` and the level in the path |
| Details | `browser.facts`, `browser.members` |